# 💳 Fraud Detection with Machine Learning

**Oasis Infobyte — Data Analytics — Level 1, Task 3**

This project detects fraudulent financial transactions in a heavily imbalanced dataset. It covers EDA, SMOTE, stratified splitting, Logistic Regression, Random Forest, Precision, Recall, F1, ROC-AUC, feature importance and scalability.

Expected dataset: `../data/creditcard.csv`. The raw CSV is not committed because it is large.

In [ ]:
import warnings
warnings.filterwarnings('ignore')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, precision_score, recall_score, f1_score, roc_auc_score, roc_curve
from imblearn.pipeline import Pipeline
from imblearn.over_sampling import SMOTE

RANDOM_STATE = 42
DATA_PATH = '../data/creditcard.csv'
if not Path(DATA_PATH).exists():
    raise FileNotFoundError('Place creditcard.csv in the data folder before running this notebook.')
df = pd.read_csv(DATA_PATH)
print('Shape:', df.shape)
display(df.head())

## 1. Data inspection and class imbalance

Accuracy is misleading here because a model can classify almost every transaction as legitimate and still appear highly accurate. Fraud detection therefore focuses on Precision, Recall, F1-score and ROC-AUC.

In [ ]:
print(df.info())
display(df.describe().T)
print('Missing values:', int(df.isna().sum().sum()))
print('Duplicates:', int(df.duplicated().sum()))
class_counts = df['Class'].value_counts().sort_index()
print(class_counts)
print(f"Fraud rate: {df['Class'].mean()*100:.4f}%")
plt.figure(figsize=(7,4))
sns.countplot(data=df, x='Class')
plt.title('Legitimate vs Fraudulent Transactions')
plt.show()

## 2. Exploratory Data Analysis

The benchmark data contains `Time` (seconds from the first transaction) and `Amount`. We inspect transaction amounts and an approximate hour-of-day pattern.

In [ ]:
plt.figure(figsize=(10,5))
sns.histplot(data=df, x='Amount', hue='Class', bins=80, element='step', stat='density', common_norm=False)
plt.xlim(0, df['Amount'].quantile(0.995))
plt.title('Transaction Amount: Fraud vs Legitimate')
plt.show()

df['Hour'] = ((df['Time'] % 86400) // 3600).astype(int)
hourly = df.groupby('Hour')['Class'].agg(['count','sum'])
hourly['fraud_rate_pct'] = hourly['sum'] / hourly['count'] * 100
plt.figure(figsize=(11,5))
sns.lineplot(data=hourly, x=hourly.index, y='fraud_rate_pct', marker='o')
plt.title('Approximate Fraud Rate by Hour')
plt.xlabel('Hour')
plt.ylabel('Fraud rate (%)')
plt.xticks(range(24))
plt.show()

## 3. Train/test split and SMOTE

Stratification ensures fraud cases occur in both sets. SMOTE is applied **only to the training data inside each pipeline**, preventing test-set leakage.

In [ ]:
X = df.drop(columns='Class')
y = df['Class']
features = X.columns.tolist()
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.20, stratify=y, random_state=RANDOM_STATE)
print('Train:', X_train.shape, 'Test:', X_test.shape)
print('Fraud in train:', int(y_train.sum()), '| Fraud in test:', int(y_test.sum()))

lr = Pipeline([('scaler', StandardScaler()), ('smote', SMOTE(random_state=RANDOM_STATE)), ('model', LogisticRegression(max_iter=1000, random_state=RANDOM_STATE))])
rf = Pipeline([('smote', SMOTE(random_state=RANDOM_STATE)), ('model', RandomForestClassifier(n_estimators=150, min_samples_leaf=2, n_jobs=-1, random_state=RANDOM_STATE))])
lr.fit(X_train, y_train)
rf.fit(X_train, y_train)
lr_pred, lr_prob = lr.predict(X_test), lr.predict_proba(X_test)[:,1]
rf_pred, rf_prob = rf.predict(X_test), rf.predict_proba(X_test)[:,1]

## 4. Model evaluation

Recall is especially important when missing fraud is costly, but high recall can increase false positives. The best threshold depends on the business cost of each error.

In [ ]:
def metrics(name, y_true, pred, prob):
    return {'Model': name, 'Precision': precision_score(y_true,pred), 'Recall': recall_score(y_true,pred), 'F1-Score': f1_score(y_true,pred), 'ROC-AUC': roc_auc_score(y_true,prob)}
results = pd.DataFrame([metrics('Logistic Regression', y_test, lr_pred, lr_prob), metrics('Random Forest', y_test, rf_pred, rf_prob)])
display(results.style.format({c:'{:.4f}' for c in results.columns if c != 'Model'}))

print('Logistic Regression\n', classification_report(y_test, lr_pred, digits=4))
print('Random Forest\n', classification_report(y_test, rf_pred, digits=4))

fig, ax = plt.subplots(1,2,figsize=(12,4))
sns.heatmap(confusion_matrix(y_test,lr_pred),annot=True,fmt='d',ax=ax[0])
ax[0].set_title('Logistic Regression')
sns.heatmap(confusion_matrix(y_test,rf_pred),annot=True,fmt='d',ax=ax[1])
ax[1].set_title('Random Forest')
plt.tight_layout(); plt.show()

lf, lt, _ = roc_curve(y_test, lr_prob); rfpr, rt, _ = roc_curve(y_test, rf_prob)
plt.figure(figsize=(8,6))
plt.plot(lf,lt,label=f'Logistic Regression AUC={roc_auc_score(y_test,lr_prob):.4f}')
plt.plot(rfpr,rt,label=f'Random Forest AUC={roc_auc_score(y_test,rf_prob):.4f}')
plt.plot([0,1],[0,1],'--')
plt.xlabel('False Positive Rate'); plt.ylabel('Recall'); plt.title('ROC-AUC Curve'); plt.legend(); plt.show()

## 5. Feature importance

Random Forest feature importance provides an interpretable view of which variables contributed most to the model's decisions.

In [ ]:
importance = pd.Series(rf.named_steps['model'].feature_importances_, index=features).sort_values(ascending=False)
plt.figure(figsize=(10,6))
sns.barplot(x=importance.head(15).values, y=importance.head(15).index)
plt.title('Top 15 Random Forest Feature Importances')
plt.show()
display(importance.head(15).to_frame('importance'))

## 6. Scalability and conclusion

One million transactions per hour is about **278 transactions per second**. A production system could use streaming ingestion, precomputed behavioural features, parallel model-serving workers, batching where appropriate, threshold tuning, monitoring and periodic retraining.

### Conclusion
Fraud detection requires more than accuracy. This project uses stratification and SMOTE to address imbalance and compares Logistic Regression with Random Forest using Precision, Recall, F1-score and ROC-AUC. In deployment, the final model and threshold should be selected using both validation performance and the financial cost of false positives and missed fraud.